# TMDB Movie Data Analysis — Data Extraction

## 1. Introduction

This notebook is the first stage of a movie data analysis project using Python, Pandas, and the TMDb API.

The objective is to collect movie metadata and credits for a predefined list of movie IDs, inspect the API responses, and prepare a raw dataset for subsequent cleaning and analysis.

### Objectives
- Connect to the TMDb API using secure authentication.
- Retrieve movie details and credits for the requested movie IDs.
- convert the API responses into a Pandas DataFrame.
- Inspect extraction results and identify failed requests.
- Save the extracted data for reproducibility.

### Tools and libraries
- Python for programming.
- Requests for HTTP communication with the API.
- Pandas for tabular data processing.
- python-dotenv for loading credentials from environment variables.

## 2. Understanding the TMDb API

The Movie Database (TMDb) provides a REST API for accessing movie metadata and related information.

This notebook uses two endpoints:

1. **Movie details:** `/movie/{movie_id}` — returns fields such as title, release date, budget, revenue, genres, runtime, and ratings.
2. **Movie credits:** `/movie/{movie_id}/credits` — returns cast and crew information, which can be used to identify directors and count cast and crew members.

Official documentation:
- [Movie Details](https://developer.themoviedb.org/reference/movie-details)
- [Movie Credits](https://developer.themoviedb.org/reference/movie-credits)
- [Authentication](https://developer.themoviedb.org/docs/authentication-application)

The details and credits endpoints are requested separately because credits are not included in the standard movie-details response.

In [1]:

from pathlib import Path
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. API Authentication

TMDb requires authentication for these requests. We will use the API Read Access Token through the HTTP Bearer authorization header.

The token is stored in a local `.env` file rather than being hardcoded in the notebook. This reduces the risk of exposing credentials when the project is published on GitHub.

The `.env` file is excluded from version control using `.gitignore`.

In [3]:

# Locate the project root.
# This works when the notebook is run from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

TMDB_TOKEN = os.getenv("TMDB_READ_ACCESS_TOKEN")

if not TMDB_TOKEN:
    raise ValueError(
        "TMDb token not found. Check the .env file "
        "in the project root."
    )

BASE_URL = "https://api.themoviedb.org/3"

print("TMDb credentials loaded successfully.")

TMDb credentials loaded successfully.


## 4. Define the Movie IDs

The project specifies 19 movie IDs to retrieve. We will preserve the supplied list, including ID `0`, so we can observe how the API handles an invalid or unavailable movie identifier.

Each valid ID will be used to request movie details and credits. Failed requests will be recorded separately so one error does not terminate the entire extraction process.

In [4]:

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597,
    135397, 420818, 24428, 168259, 99861,
    284054, 12445, 181808, 330457, 351286,
    109445, 321612, 260513
]

print(f"Number of requested IDs: {len(movie_ids)}")
print(f"Unique requested IDs: {len(set(movie_ids))}")

Number of requested IDs: 19
Unique requested IDs: 19


## 5. Configure HTTP Requests

We create a reusable HTTP session and attach the Bearer token to its headers. Reusing the session avoids repeating authentication setup for every request.

The extraction function will request details and credits separately. It will check HTTP responses and record errors for later inspection.

In [5]:

session = requests.Session()

session.headers.update({
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
})

print("HTTP session configured.")
  

HTTP session configured.


## 6. Extract Movie Details and Credits

The function below requests movie metadata and credits for a single movie ID. The metadata response contains the primary movie attributes, while the credits response provides cast and crew details.

We derive the following fields from the credits:
- `cast`: list of cast member names.
- `cast_size`: number of cast entries returned by the API.
- `director`: names of crew members whose job is `Director`.
- `crew_size`: number of crew entries returned by the API.

The function returns the movie record and any error information separately. This allows the remaining IDs to be processed even if an individual request fails.

In [6]:

def fetch_movie(movie_id):
    """Fetch one movie's details and credits from TMDb."""

    details_url = f"{BASE_URL}/movie/{movie_id}"
    credits_url = f"{BASE_URL}/movie/{movie_id}/credits"

    try:
        details_response = session.get(
            details_url,
            params={"language": "en-US"},
            timeout=30
        )
        details_response.raise_for_status()
        movie = details_response.json()

        credits_response = session.get(
            credits_url,
            timeout=30
        )
        credits_response.raise_for_status()
        credits = credits_response.json()

        cast = credits.get("cast", [])
        crew = credits.get("crew", [])

        directors = [
            person["name"]
            for person in crew
            if person.get("job") == "Director"
            and person.get("name")
        ]

        movie["cast"] = [
            person["name"]
            for person in cast
            if person.get("name")
        ]
        movie["cast_size"] = len(cast)
        movie["director"] = " | ".join(directors) or None
        movie["crew_size"] = len(crew)

        return movie, None

    except requests.HTTPError as exc:
        status_code = exc.response.status_code

        return None, {
            "movie_id": movie_id,
            "status_code": status_code,
            "error": str(exc)
        }

    except (requests.RequestException, ValueError) as exc:
        return None, {
            "movie_id": movie_id,
            "status_code": None,
            "error": str(exc)
        }


## 7. Execute the Extraction

We now loop through the supplied movie IDs and call the extraction function for each one.

Successful records are stored in a list, while errors are recorded separately. We include a short delay between movies to avoid sending requests unnecessarily quickly.

The extraction is successful only when a movie-details request and its corresponding credits request both succeed.

In [7]:

movies = []
errors = []

for position, movie_id in enumerate(movie_ids, start=1):
    movie, error = fetch_movie(movie_id)

    if movie is not None:
        movies.append(movie)
        print(
            f"[{position}/{len(movie_ids)}] "
            f"Fetched: {movie.get('title', 'Unknown title')}"
        )
    else:
        errors.append(error)
        print(
            f"[{position}/{len(movie_ids)}] "
            f"Failed ID {movie_id}: {error['error']}"
        )

    time.sleep(0.25)

print("\nExtraction summary")
print("------------------")
print(f"Requested: {len(movie_ids)}")
print(f"Successful: {len(movies)}")
print(f"Failed: {len(errors)}")


[1/19] Failed ID 0: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/0?language=en-US
[2/19] Fetched: Avengers: Endgame
[3/19] Fetched: Avatar
[4/19] Fetched: Star Wars: The Force Awakens
[5/19] Fetched: Avengers: Infinity War
[6/19] Fetched: Titanic
[7/19] Fetched: Jurassic World
[8/19] Fetched: The Lion King
[9/19] Fetched: The Avengers
[10/19] Fetched: Furious 7
[11/19] Fetched: Avengers: Age of Ultron
[12/19] Fetched: Black Panther
[13/19] Fetched: Harry Potter and the Deathly Hallows: Part 2
[14/19] Fetched: Star Wars: The Last Jedi
[15/19] Fetched: Frozen II
[16/19] Fetched: Jurassic World: Fallen Kingdom
[17/19] Fetched: Frozen
[18/19] Fetched: Beauty and the Beast
[19/19] Fetched: Incredibles 2

Extraction summary
------------------
Requested: 19
Successful: 18
Failed: 1


## 8. Create the Raw Movie DataFrame

Each successful API response is a dictionary containing movie attributes. Pandas can combine these dictionaries into a tabular DataFrame, automatically creating columns for the returned fields.

At this stage, we preserve the original nested structures, such as genres, production companies, and spoken languages. We will flatten and clean these fields in the next notebook.

We also create a separate DataFrame for failed requests to make troubleshooting easier.

In [8]:

df = pd.DataFrame(movies)
errors_df = pd.DataFrame(errors)

print("Dataset dimensions:", df.shape)
print("Column count:", len(df.columns))

display(df.head())


Dataset dimensions: (18, 31)
Column count: 31


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,status,tagline,title,video,vote_average,vote_count,cast,cast_size,director,crew_size
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,...,Released,Avenge the fallen.,Avengers: Endgame,False,8.246,28999,"[Robert Downey Jr., Chris Evans, Mark Ruffalo,...",103,Joe Russo | Anthony Russo,613
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,...,Released,Enter the world of Pandora.,Avatar,False,7.600,34836,"[Sam Worthington, Zoe Saldaña, Sigourney Weave...",68,James Cameron,999
2,False,/8BTsTfln4jlQrLXUBquXJ0ASQy9.jpg,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],en,...,Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.200,21007,"[Daisy Ridley, John Boyega, Oscar Isaac, Adam ...",99,J.J. Abrams,270
3,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",300000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",https://www.marvel.com/movies/avengers-infinit...,299536,tt4154756,[US],en,...,Released,An entire universe. Once and for all.,Avengers: Infinity War,False,8.200,33143,"[Robert Downey Jr., Chris Evans, Chris Hemswor...",69,Joe Russo | Anthony Russo,738
4,False,/xXCuto8YVp5RFqBJ7yKmVmLOWpF.jpg,None,200000000,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",https://www.paramountmovies.com/movies/titanic,597,tt0120338,[US],en,...,Released,Nothing on earth could come between them.,Titanic,False,7.901,27894,"[Leonardo DiCaprio, Kate Winslet, Billy Zane, ...",116,James Cameron,271


## 9. Initial Data Inspection

Before saving the dataset, we inspect the returned movie IDs, titles, release dates, budgets, revenues, and ratings.

These checks help us confirm that the API data is structured as expected and identify missing or duplicate identifiers. They are preliminary checks only; detailed validation and cleaning will be performed in the next stage.

In [13]:

columns_to_inspect = [
    "id", "title", "release_date",
    "budget", "revenue", "vote_average"
]

available_columns = [
    column for column in columns_to_inspect
    if column in df.columns
]

display(df[available_columns].head(10))

if "id" in df.columns:
    print("Unique returned IDs:", df["id"].nunique())
    print("Duplicate returned IDs:", df["id"].duplicated().sum())

print("Missing titles:", df["title"].isna().sum())
display(errors_df)

,id,title,release_date,budget,revenue,vote_average
0,299534,Avengers: Endgame,2019-04-24,356000000,2927449455,8.246
1,19995,Avatar,2009-12-16,237000000,2923706026,7.600
2,140607,Star Wars: The Force Awakens,2015-12-15,245000000,2068223624,7.200
3,299536,Avengers: Infinity War,2018-04-25,300000000,2052415039,8.200
4,597,Titanic,1997-12-18,200000000,2264162353,7.901
5,135397,Jurassic World,2015-06-06,150000000,1671537444,6.703
6,420818,The Lion King,2019-07-12,260000000,1662020819,7.090
7,24428,The Avengers,2012-04-25,220000000,1518815515,8.080
8,168259,Furious 7,2015-04-01,190000000,1515400000,7.209
9,99861,Avengers: Age of Ultron,2015-04-22,235000000,1405403694,7.278


Unique returned IDs: 18
Duplicate returned IDs: 0
Missing titles: 0


,movie_id,status_code,error
0,0,404,404 Client Error: Not Found for url: https://a...


## 10. Save the Extraction Results

We save the extracted movie records and any failed-request details in the project's `data/raw/` directory.

CSV is convenient for inspection, but the original API responses contain nested JSON structures. Pandas will serialize these nested values into strings in the CSV file. For a more faithful copy of the API data, we can also save the records as JSON.

The raw data directory is excluded from Git so the generated data does not clutter the repository.

In [14]:

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(
    raw_dir / "tmdb_movies_raw.csv",
    index=False
)

df.to_json(
    raw_dir / "tmdb_movies_raw.json",
    orient="records",
    indent=2,
    force_ascii=False
)

errors_df.to_csv(
    raw_dir / "tmdb_fetch_errors.csv",
    index=False
)

print("Extraction files saved in:", raw_dir)


Extraction files saved in: d:\AmaliTech\TMDB-Movie-Analysis\data\raw


## 11. Extraction Summary and Conclusion

In this notebook, we configured TMDb authentication, retrieved movie details and credits, created a Pandas DataFrame, inspected the extraction results, and saved the raw data.

### Results to record

After executing the notebook, document:
- Total movie IDs requested.
- Number of successfully extracted movies.
- Number of failed requests and their causes.
- DataFrame dimensions and any initial anomalies.

### Next step

The next notebook will clean and preprocess the raw dataset, extract values from nested JSON columns, standardize data types, handle missing values, and prepare the final analysis DataFrame.